# Week 7 — Tool Agent Development & Data Integration
### Real-Time Urban Data Integration Platform
**Team 4** | Kritika Gupta | Rohit Manohar Saggam | Nandini Devi Poreddy

**Modules covered in this notebook:**
1. Dynamic Location-Based Retrieval
2. Weather Tool Agent (Open-Meteo API)
3. AQI Tool Agent (OpenAQ API)
4. Unified Data Integration Layer
5. Intent Classification Module

## Step 1: Install Required Libraries

In [ ]:
!pip install requests -q
print('Libraries installed successfully.')

## Step 2: Dynamic Location-Based Retrieval
**Owner: Nandini**

Converts city names or user text into standardized geographic coordinates (latitude & longitude).
Uses the Open-Meteo Geocoding API — no API key required.

In [ ]:
import requests
import json

def get_coordinates(city: str) -> dict:
    """
    Converts a city name to latitude and longitude coordinates.

    Args:
        city (str): Name of the city (e.g., 'Toronto', 'Mumbai')

    Returns:
        dict: City info with latitude, longitude, and country
    """
    try:
        url = 'https://geocoding-api.open-meteo.com/v1/search'
        params = {'name': city, 'count': 1, 'language': 'en', 'format': 'json'}
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if 'results' not in data or len(data['results']) == 0:
            return {'error': f"City '{city}' not found. Please check the spelling."}

        loc = data['results'][0]
        return {
            'city': loc['name'],
            'country': loc.get('country', ''),
            'latitude': loc['latitude'],
            'longitude': loc['longitude'],
            'timezone': loc.get('timezone', '')
        }
    except requests.exceptions.Timeout:
        return {'error': 'Request timed out. Please try again.'}
    except Exception as e:
        return {'error': f'Unexpected error: {str(e)}'}


# --- Test: Single city ---
print('=== Location Retrieval Test ===')
result = get_coordinates('Toronto')
print(json.dumps(result, indent=2))

# --- Test: Multiple cities ---
print('\n=== Multiple City Test ===')
cities = ['London', 'Tokyo', 'Mumbai', 'InvalidCityXYZ']
for city in cities:
    coords = get_coordinates(city)
    if 'error' in coords:
        print(f'{city}: ERROR - {coords["error"]}')
    else:
        print(f'{city}: lat={coords["latitude"]}, lon={coords["longitude"]}, country={coords["country"]}')

## Step 3: Weather Tool Agent
**Owner: Kritika**

Fetches real-time weather data for a given city using the Open-Meteo API.
Extracts temperature, humidity, wind speed, and weather conditions.

In [ ]:
def get_weather(city: str) -> dict:
    """
    Fetches real-time weather data for a given city.

    Args:
        city (str): Name of the city

    Returns:
        dict: Structured weather data including temperature, humidity, wind speed
    """
    # Step 1: Get coordinates
    coords = get_coordinates(city)
    if 'error' in coords:
        return coords

    lat = coords['latitude']
    lon = coords['longitude']

    # WMO Weather Code descriptions
    weather_descriptions = {
        0: 'Clear sky', 1: 'Mainly clear', 2: 'Partly cloudy', 3: 'Overcast',
        45: 'Foggy', 48: 'Icy fog',
        51: 'Light drizzle', 53: 'Moderate drizzle', 55: 'Dense drizzle',
        61: 'Slight rain', 63: 'Moderate rain', 65: 'Heavy rain',
        71: 'Slight snow', 73: 'Moderate snow', 75: 'Heavy snow',
        80: 'Slight showers', 81: 'Moderate showers', 82: 'Violent showers',
        95: 'Thunderstorm', 96: 'Thunderstorm with hail', 99: 'Thunderstorm with heavy hail'
    }

    try:
        # Step 2: Fetch weather data
        url = 'https://api.open-meteo.com/v1/forecast'
        params = {
            'latitude': lat,
            'longitude': lon,
            'current_weather': True,
            'hourly': 'temperature_2m,relativehumidity_2m,apparent_temperature',
            'forecast_days': 1
        }
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if 'current_weather' not in data:
            return {'error': 'Weather data unavailable for this location.'}

        current = data['current_weather']
        hourly = data.get('hourly', {})
        humidity = hourly.get('relativehumidity_2m', [None])[0]
        feels_like = hourly.get('apparent_temperature', [None])[0]
        weathercode = current['weathercode']

        return {
            'city': coords['city'],
            'country': coords['country'],
            'latitude': lat,
            'longitude': lon,
            'temperature_celsius': current['temperature'],
            'temperature_fahrenheit': round(current['temperature'] * 9/5 + 32, 1),
            'feels_like_celsius': feels_like,
            'humidity_percent': humidity,
            'wind_speed_kmh': current['windspeed'],
            'weather_condition': weather_descriptions.get(weathercode, 'Unknown'),
            'weather_code': weathercode,
            'data_source': 'Open-Meteo API'
        }

    except requests.exceptions.Timeout:
        return {'error': 'Request timed out. Please try again.'}
    except Exception as e:
        return {'error': f'Unexpected error: {str(e)}'}


# --- Test Weather Agent ---
print('=== Weather Tool Agent Test ===')
cities = ['New York', 'London', 'Tokyo']
for city in cities:
    print(f'\n--- {city} ---')
    result = get_weather(city)
    if 'error' in result:
        print(f'ERROR: {result["error"]}')
    else:
        print(f'Temperature  : {result["temperature_celsius"]}C / {result["temperature_fahrenheit"]}F')
        print(f'Feels Like   : {result["feels_like_celsius"]}C')
        print(f'Humidity     : {result["humidity_percent"]}%')
        print(f'Wind Speed   : {result["wind_speed_kmh"]} km/h')
        print(f'Conditions   : {result["weather_condition"]}')

## Step 4: AQI Tool Agent
**Owner: Rohit**

Fetches real-time Air Quality Index (AQI) and pollutant data.
Uses the Open-Meteo Air Quality API — no API key required.
Extracts PM2.5, PM10, and computes AQI health category.

In [ ]:
def get_aqi_category(pm25: float) -> dict:
    """
    Returns AQI category and health recommendation based on PM2.5 value.

    Args:
        pm25 (float): PM2.5 concentration in µg/m³

    Returns:
        dict: AQI category, color, and health recommendation
    """
    if pm25 <= 12:
        return {'category': 'Good', 'color': 'Green', 'aqi_range': '0-50',
                'recommendation': 'Air quality is satisfactory. Safe for all activities.'}
    elif pm25 <= 35.4:
        return {'category': 'Moderate', 'color': 'Yellow', 'aqi_range': '51-100',
                'recommendation': 'Acceptable air quality. Sensitive groups should take care.'}
    elif pm25 <= 55.4:
        return {'category': 'Unhealthy for Sensitive Groups', 'color': 'Orange', 'aqi_range': '101-150',
                'recommendation': 'Sensitive groups should limit prolonged outdoor activity.'}
    elif pm25 <= 150.4:
        return {'category': 'Unhealthy', 'color': 'Red', 'aqi_range': '151-200',
                'recommendation': 'Everyone should reduce prolonged outdoor exertion.'}
    elif pm25 <= 250.4:
        return {'category': 'Very Unhealthy', 'color': 'Purple', 'aqi_range': '201-300',
                'recommendation': 'Avoid outdoor activities. Health alert for everyone.'}
    else:
        return {'category': 'Hazardous', 'color': 'Maroon', 'aqi_range': '301+',
                'recommendation': 'Emergency conditions. Stay indoors immediately.'}


def get_aqi(city: str) -> dict:
    """
    Fetches real-time AQI and pollutant data for a given city.

    Args:
        city (str): Name of the city

    Returns:
        dict: AQI data including PM2.5, PM10, and health category
    """
    # Step 1: Get coordinates
    coords = get_coordinates(city)
    if 'error' in coords:
        return coords

    lat = coords['latitude']
    lon = coords['longitude']

    try:
        # Step 2: Fetch AQI data from Open-Meteo Air Quality API
        url = 'https://air-quality-api.open-meteo.com/v1/air-quality'
        params = {
            'latitude': lat,
            'longitude': lon,
            'current': 'pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,ozone,dust',
            'timezone': 'auto'
        }
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if 'current' not in data:
            return {'error': 'AQI data unavailable for this location.'}

        current = data['current']
        pm25 = current.get('pm2_5', 0)
        pm10 = current.get('pm10', 0)
        no2 = current.get('nitrogen_dioxide', 0)
        o3 = current.get('ozone', 0)
        co = current.get('carbon_monoxide', 0)

        # Get AQI category
        aqi_info = get_aqi_category(pm25 or 0)

        return {
            'city': coords['city'],
            'country': coords['country'],
            'latitude': lat,
            'longitude': lon,
            'pm2_5_ugm3': pm25,
            'pm10_ugm3': pm10,
            'nitrogen_dioxide_ugm3': no2,
            'ozone_ugm3': o3,
            'carbon_monoxide_ugm3': co,
            'aqi_category': aqi_info['category'],
            'aqi_color': aqi_info['color'],
            'aqi_range': aqi_info['aqi_range'],
            'health_recommendation': aqi_info['recommendation'],
            'data_source': 'Open-Meteo Air Quality API'
        }

    except requests.exceptions.Timeout:
        return {'error': 'Request timed out. Please try again.'}
    except Exception as e:
        return {'error': f'Unexpected error: {str(e)}'}


# --- Test AQI Agent ---
print('=== AQI Tool Agent Test ===')
cities = ['Delhi', 'Paris', 'Sydney']
for city in cities:
    print(f'\n--- {city} ---')
    result = get_aqi(city)
    if 'error' in result:
        print(f'ERROR: {result["error"]}')
    else:
        print(f'PM2.5        : {result["pm2_5_ugm3"]} µg/m³')
        print(f'PM10         : {result["pm10_ugm3"]} µg/m³')
        print(f'NO2          : {result["nitrogen_dioxide_ugm3"]} µg/m³')
        print(f'AQI Category : {result["aqi_category"]} ({result["aqi_color"]})')
        print(f'Health Note  : {result["health_recommendation"]}')

## Step 5: Unified Data Integration Layer
**Owner: Rohit**

Combines outputs from both Weather and AQI tool agents into a single, consistent JSON structure.
Handles partial failures gracefully if one API is unavailable.

In [ ]:
def get_combined_data(city: str) -> dict:
    """
    Fetches and combines weather + AQI data for a given city.

    Args:
        city (str): Name of the city

    Returns:
        dict: Unified JSON with weather and AQI data
    """
    print(f'Fetching data for {city}...')

    weather_data = get_weather(city)
    aqi_data = get_aqi(city)

    # Determine overall status
    weather_ok = 'error' not in weather_data
    aqi_ok = 'error' not in aqi_data

    if weather_ok and aqi_ok:
        status = 'success'
    elif weather_ok or aqi_ok:
        status = 'partial_success'
    else:
        status = 'failed'

    return {
        'city': city,
        'status': status,
        'weather': weather_data if weather_ok else {'error': weather_data.get('error')},
        'air_quality': aqi_data if aqi_ok else {'error': aqi_data.get('error')},
        'data_available': {
            'weather': weather_ok,
            'air_quality': aqi_ok
        }
    }


# --- Test Unified Data Integration ---
print('=== Unified Data Integration Test ===')
test_city = 'Chicago'
combined = get_combined_data(test_city)

print(f'\nStatus: {combined["status"]}')
print(f'Weather Available: {combined["data_available"]["weather"]}')
print(f'AQI Available: {combined["data_available"]["air_quality"]}')

if combined['data_available']['weather']:
    w = combined['weather']
    print(f'\nWeather: {w["temperature_celsius"]}C, {w["weather_condition"]}')

if combined['data_available']['air_quality']:
    a = combined['air_quality']
    print(f'AQI: {a["aqi_category"]} - {a["health_recommendation"]}')

## Step 6: Intent Classification Module
**Owner: Kritika**

Classifies user queries into:
- `weather_only` — user wants weather info
- `aqi_only` — user wants air quality info
- `combined` — user wants both
- `comparison` — user wants to compare two cities
- `unknown` — query is unclear

In [ ]:
import re

def classify_intent(query: str) -> dict:
    """
    Classifies user query intent for routing to the correct tool agent.

    Args:
        query (str): Natural language query from the user

    Returns:
        dict: Intent classification with detected cities
    """
    query_lower = query.lower()

    # Weather keywords
    weather_keywords = [
        'weather', 'temperature', 'temp', 'rain', 'wind', 'humid',
        'sunny', 'cloudy', 'snow', 'hot', 'cold', 'warm', 'forecast',
        'feels like', 'celsius', 'fahrenheit'
    ]

    # AQI keywords
    aqi_keywords = [
        'aqi', 'air quality', 'pollution', 'pm2.5', 'pm10',
        'smog', 'pollutant', 'nitrogen', 'ozone', 'breathe',
        'air', 'toxic', 'hazardous'
    ]

    # Comparison keywords
    comparison_keywords = ['compare', 'vs', 'versus', 'difference between', 'better than']

    has_weather = any(kw in query_lower for kw in weather_keywords)
    has_aqi = any(kw in query_lower for kw in aqi_keywords)
    has_comparison = any(kw in query_lower for kw in comparison_keywords)

    # Determine intent
    if has_comparison:
        intent = 'comparison'
    elif has_weather and has_aqi:
        intent = 'combined'
    elif has_aqi:
        intent = 'aqi_only'
    elif has_weather:
        intent = 'weather_only'
    else:
        # Default to combined if city is mentioned but no specific keyword
        intent = 'combined'

    return {
        'intent': intent,
        'has_weather_keywords': has_weather,
        'has_aqi_keywords': has_aqi,
        'has_comparison': has_comparison,
        'original_query': query
    }


# --- Test Intent Classification ---
print('=== Intent Classification Test ===')
test_queries = [
    'What is the temperature in Toronto?',
    'How is the air quality in Delhi today?',
    'Give me weather and AQI for London',
    'Compare air pollution between Chicago and Detroit',
    'Is it safe to go outside in Mumbai?',
    'What is the PM2.5 level in Beijing?'
]

for query in test_queries:
    result = classify_intent(query)
    print(f'Query  : {query}')
    print(f'Intent : {result["intent"]}')
    print()

## Step 7: Full Week 7 Pipeline Test

Testing the complete flow: Intent Classification → Tool Agent Selection → Data Retrieval → Output

In [ ]:
def week7_pipeline(user_query: str) -> None:
    """
    Full Week 7 pipeline: classify intent, route to correct agent, display results.

    Args:
        user_query (str): Natural language query
    """
    print('=' * 60)
    print(f'Query: {user_query}')
    print('=' * 60)

    # Step 1: Classify intent
    intent_result = classify_intent(user_query)
    intent = intent_result['intent']
    print(f'Intent Detected: {intent}')

    # Step 2: Extract city from query (simple extraction)
    # In a real system, DeepSeek would handle this
    words = user_query.replace('?', '').split()
    city = words[-1]  # Simple heuristic: last word is usually the city
    print(f'City Detected  : {city}')
    print('-' * 60)

    # Step 3: Route to correct agent
    if intent == 'weather_only':
        data = get_weather(city)
        if 'error' not in data:
            print(f'Temperature  : {data["temperature_celsius"]}C / {data["temperature_fahrenheit"]}F')
            print(f'Feels Like   : {data["feels_like_celsius"]}C')
            print(f'Humidity     : {data["humidity_percent"]}%')
            print(f'Wind Speed   : {data["wind_speed_kmh"]} km/h')
            print(f'Conditions   : {data["weather_condition"]}')

    elif intent == 'aqi_only':
        data = get_aqi(city)
        if 'error' not in data:
            print(f'PM2.5        : {data["pm2_5_ugm3"]} µg/m³')
            print(f'PM10         : {data["pm10_ugm3"]} µg/m³')
            print(f'AQI Category : {data["aqi_category"]} ({data["aqi_color"]})')
            print(f'Health Note  : {data["health_recommendation"]}')

    else:  # combined or unknown
        data = get_combined_data(city)
        if data['data_available']['weather']:
            w = data['weather']
            print(f'--- Weather ---')
            print(f'Temperature  : {w["temperature_celsius"]}C / {w["temperature_fahrenheit"]}F')
            print(f'Conditions   : {w["weather_condition"]}')
        if data['data_available']['air_quality']:
            a = data['air_quality']
            print(f'--- Air Quality ---')
            print(f'PM2.5        : {a["pm2_5_ugm3"]} µg/m³')
            print(f'AQI Category : {a["aqi_category"]} ({a["aqi_color"]})')
            print(f'Health Note  : {a["health_recommendation"]}')

    print()


# --- Run Pipeline Tests ---
week7_pipeline('What is the temperature in Toronto?')
week7_pipeline('How is the air quality in Delhi?')
week7_pipeline('Give me full environment report for London')